In [1]:
import sys
import os 
import numpy as np
import pandas as pd
import re
from pulp import *

sys.path.append(os.path.abspath("../../")) ; from optimizer import variables
sys.path.insert(0, os.path.abspath("../../../Generic-Parallel-Compute-Helper/")); from parallel_compute import *
from functools import partial
from tqdm import tqdm

pd.set_option('display.max_rows', None) ; pd.set_option('display.max_columns', None)

## Constraints:

#### Ballance PV output

In [2]:
def _interval_constraint_A(Lp, gen, pv2grid, pv2battery, curtailment, i):
    
    Lp += pv2grid + pv2battery + curtailment == gen, f"Ballance_PV_output_{i}"
    
    return Lp

#### Ballance BESS charge

In [3]:
def _interval_constraint_B(Lp, pv2battery_eff, grid2battery_eff, dischargeE, previous_interval_mwh, stored_charge,i):
   
    Lp += previous_interval_mwh + (pv2battery_eff + grid2battery_eff) - dischargeE == stored_charge, f"Ballance_BESS_charge_{i}"
    
    return Lp

#### Restrict BESS by BESS capacity

In [4]:
def _interval_constraint_C(Lp, pv2battery_eff, grid2battery_eff, dischargeE_eff, bess_energy_limit_per_interval_mw,i):

    Lp += pv2battery_eff + grid2battery_eff + dischargeE_eff <= bess_energy_limit_per_interval_mw, f"Restrict_BESS_by_BESS_capacity_{i}"

    return Lp

#### Restrict BESS by BESS physical limits

In [5]:
def _interval_constraint_D(Lp, stored_charge, bess_max_eff, chargeE, dischargeE, gen_exportable_now, pv2battery, gen_exportable,i):

    Lp += stored_charge <= bess_max_eff, f"Restrict_stored_charge_by_physcial_limit_{i}"

    Lp += chargeE <= bess_max_eff, f"Restrict_chargeE_by_physcial_limit_{i}"

    Lp += dischargeE <= bess_max_eff, f"Restrict_dischargeE_by_physcial_limit_{i}"

    # if gen_exportable_now == True:
    #     Lp += pv2battery <= gen_exportable
            
    return Lp

#### Restrict grid by grid capacity 

In [6]:
def _interval_constraint_E(Lp, pv2grid, dischargeE_eff, project_poi_export_limit_mwh_per_interval, grid2battery, project_poi_import_limit_mwh_per_interval,i):

  
    Lp += pv2grid + dischargeE_eff + grid2battery <= project_poi_export_limit_mwh_per_interval, f"Restrict_grid_by_grid_capacity_{i}"

    # Lp += grid2battery <= project_poi_import_limit_mwh_per_interval
    
    return Lp

#### Restrict BESS by signal mutuality

In [7]:
def _interval_constraint_F(Lp, charge_signal, discharge_signal, chargeE, dischargeE,i):

    if charge_signal == True:
        Lp += dischargeE == 0, f"Restrict_BESS_by_dischargeE_mutuality_{i}"
    
    if discharge_signal == True:
        Lp += chargeE == 0, f"Restrict_BESS_by_chargeE_mutuality_{i}"

    return Lp


#### Restrict grid importing during sun soaker times

In [8]:
def _interval_constraint_G(Lp, sun_soaker, grid2battery,i):
    
    if sun_soaker == True:
        Lp += grid2battery == 0,f"Restrict_importing_by_sunsoaker_{i}"
    
    return Lp

#### Restrict imports and exports to be equal or less than max monthly values

In [9]:
def _interval_constraint_H(Lp, tou_label, grid2battery, imports_max_peak, imports_max_shoulder, imports_max_offpeak, exports_max_sunsoaker, dischargeE_eff, pv2grid,i): 

        if tou_label == "Peak":
            Lp += grid2battery <= imports_max_peak, f"Restrict_onpeak_grid_imports_by_max_{i}"

        elif tou_label == "Shoulder":
            Lp += grid2battery <= imports_max_shoulder, f"Restrict_shoulder_grid_imports_by_max_{i}"

        elif tou_label == "Off-peak":
            Lp += grid2battery <= imports_max_offpeak, f"Restrict_peak_grid_imports_by_max_{i}"

        elif tou_label == "Sun Soaker":
            Lp += pv2grid + dischargeE_eff <= exports_max_sunsoaker, f"Restrict_sunsoaker_grid_exports_by_max_{i}"

        return Lp


#### Restrict BESS by cycle limit

In [10]:
def _window_constraint_A(Lp, window_cycle_budget_mwh, total_window_discharge):

    Lp += total_window_discharge <= window_cycle_budget_mwh, f"Restrict_BESS_by_cycle_limit"

    return Lp

## Panalties:

#### Penalise revenue by, price of, remaining BESS headroom, upon charge signal

In [11]:
def _interval_penalty_A(bess_capacity_remaining, charge_signal, trade_signal_penalty):
    
    if charge_signal:
        charge_signal_penalty = bess_capacity_remaining * trade_signal_penalty
        
        return charge_signal_penalty


#### Penalise revenue by, price of, remaining BESS stored charge, upon discharge signal

In [12]:
def _interval_penalty_B(discharge_signal, trade_signal_penalty, stored_charge):
   
    if discharge_signal:
        discharge_signal_penalty = stored_charge * trade_signal_penalty

        return discharge_signal_penalty


#### Penalise revenue by, price of, curtailed energy

In [13]:
def _interval_penalty_C(curtailment, curtailment_penalty):
    curtailed_energy_penalty = curtailment * curtailment_penalty
    
    return curtailed_energy_penalty

#### Penalise revenue by, import/export tarrifs

In [14]:
def _monthly_penalty_A(import_charge_peak, import_charge_shoulder, import_charge_off_peak, export_charge_sun_soaker, grid2battery_max_peak, grid2battery_max_shoulder, grid2battery_max_offpeak, export_max):
        
    demand_penalty = (import_charge_peak * grid2battery_max_peak) + (import_charge_shoulder * grid2battery_max_shoulder) + (import_charge_off_peak * grid2battery_max_offpeak) + (export_charge_sun_soaker * export_max)
    
    return demand_penalty

## Objective expression - interval level:

#### Each interval builds one symbolic revenue expression; the final objective sums them once.



In [15]:
def _interval_expression(price, grid_import_penalty, lgc_price, grid2battery, pv2grid, dischargeE_eff, import_charge, charge_signal, discharge_signal, charge_signal_penalty, discharge_signal_penalty, curtailed_energy_penalty):

    objective_components = {
        "exporting_energy_revenue": (pv2grid + dischargeE_eff) * price,
        "lgc_revenue": (pv2grid + dischargeE_eff - grid2battery * grid_import_penalty) * lgc_price,
        "importing_energy_cost": -(grid2battery * (price + import_charge)),
        "curtailed_energy_penalty": -curtailed_energy_penalty,
        "charge_signal_penalty": -charge_signal_penalty if charge_signal else 0,
        "discharge_signal_penalty": -discharge_signal_penalty if discharge_signal else 0,
    }

    interval_expression = lpSum(objective_components.values())

    return interval_expression, objective_components


## Objective expression - window level:

In [16]:
def _objective(self):
    
    self.Lp += lpSum(self.interval_objective_expressions) - lpSum(self.monthly_import_export_penalties), "Total_window_revenue"

    return self.Lp

## Within-window optimisation design logic:

### Export logic

In [17]:
def export_results(self):
    """Package window optimization results.

    Top-level dispatch:
      run_multiple_optimisations == False  →  full diagnostic export
      run_multiple_optimisations == True   →  lean export for parallel/batch runs

    Within the full export, revenue semantics depend on the optimisation directive
    and predicted_price_type:
      directive 1  (perfect foresight)     : LP used actual prices → actual_revenue == objective
      directive 2, predicted_price_type 1  : predicted prices only
      directive 2, predicted_price_type 2  : BESS instructions only (LP price = 0)
      directive 2, predicted_price_type 3  : predicted prices + BESS instructions
    """

    # ------------------------------------------------------------------ #
    # Shared helper                                                        #
    # ------------------------------------------------------------------ #

    def _compute_actual_revenue():
        """Energy trading P&L at actual spot prices: export revenue + LGC - import costs.
        Does not include demand charges or LP penalties — those are added separately in
        _add_revenue_columns so the final actual_revenue_row_total is total realised P&L."""
        n = len(self.dataset_window_subset)
        actual_price   = self.dataset_window_subset["Price"].to_numpy()
        lgc            = self.dataset_window_subset["LGC"].to_numpy()
        import_charge  = self.dataset_window_subset["Import_charge"].to_numpy()
        hte2grid       = self.dataset_window_subset["hte2grid"].to_numpy()

        pv2grid_v      = np.array([value(v) or 0.0 for v in self.pv2grid[:n]])
        dischargeE_v   = np.array([value(v) or 0.0 for v in self.dischargeE[:n]])
        grid2battery_v = np.array([value(v) or 0.0 for v in self.grid2battery[:n]])
        dischargeE_eff = dischargeE_v * hte2grid

        return (
            (pv2grid_v + dischargeE_eff) * actual_price
            + (pv2grid_v + dischargeE_eff - grid2battery_v * variables.grid_import_penalty) * lgc
            - grid2battery_v * (actual_price + import_charge)
        )

    # ------------------------------------------------------------------ #
    # Full export — single optimisation                                   #
    # ------------------------------------------------------------------ #

    def _export_single():
        """Full diagnostic DataFrame for a single (non-parallel) run.
        Contains: input columns, decision variables, objective components,
        constraint diagnostics (value / slack / dual), and revenue totals."""

        base_df = self.dataset_window_subset.copy().reset_index(drop=True)
        base_df.columns = [f"input_{col}" for col in base_df.columns]
        n = len(base_df)

        # ---- variable name parsers ---- #

        def _split_interval_name(name):
            m = re.match(r"^(.*)_(\d+)$", name)
            if not m:
                return name, None
            base, suffix = m.group(1), int(m.group(2))
            if 0 <= suffix < n and not re.search(r"_\d{4}$", base):
                return base, suffix
            return name, None

        def _split_monthly_name(name):
            m = re.match(r"^(.*)_(\d{4})_(\d{2})$", name)
            if not m:
                return name, None
            return m.group(1), (int(m.group(2)), int(m.group(3)))

        def _month_rows(ym):
            return [idx for idx, row_ym in enumerate(self.window_year_month) if row_ym == ym]

        cols = {}

        # ---- decision variables ---- #

        def _extract_decision_variables():
            for var in self.Lp.variables():
                base, i = _split_interval_name(var.name)
                if i is not None:
                    cols.setdefault(f"decision_variable_{base}", [None] * n)[i] = value(var)
                    continue

                base, ym = _split_monthly_name(var.name)
                if ym is not None:
                    col = f"decision_variable_{base}"
                    cols.setdefault(col, [None] * n)
                    for row_i in _month_rows(ym):
                        cols[col][row_i] = value(var)
                    continue

                cols[f"scalar_decision_variable_{var.name}"] = [value(var)] * n

        # ---- objective components ---- #

        def _extract_objective_components():
            for component_name, interval_exprs in self.objective_component_map.items():
                col = f"objective_component_{component_name}"
                cols[col] = [0.0] * n
                for i, expr in interval_exprs.items():
                    if i is not None and 0 <= i < n:
                        cols[col][i] += value(expr)

        # ---- constraint diagnostics ---- #
        # value = evaluated constraint expression after solve
        # slack = unused room in the constraint ("how much spare room?")
        # dual  = shadow price / marginal value of relaxing that constraint

        def _extract_constraint_diagnostics():
            for cname, c in self.Lp.constraints.items():
                i = self.constraint_map.get(cname)
                if i is None:
                    cols[f"scalar_constraint_value_{cname}"] = [value(c)] * n
                    cols[f"scalar_constraint_slack_{cname}"] = [c.slack] * n
                    cols[f"scalar_constraint_dual_{cname}"] = [c.pi] * n
                    continue

                base = cname.rsplit("_", 1)[0]
                cols.setdefault(f"constraint_value_{base}", [None] * n)[i] = value(c)
                cols.setdefault(f"constraint_slack_{base}", [None] * n)[i] = c.slack
                cols.setdefault(f"constraint_dual_{base}", [None] * n)[i] = c.pi

        _extract_decision_variables()
        _extract_objective_components()
        _extract_constraint_diagnostics()

        extra_df = pd.DataFrame(cols)

        component_cols = [c for c in extra_df.columns if c.startswith("objective_component_")]
        extra_df["objective_row_total"] = extra_df[component_cols].sum(axis=1) if component_cols else 0.0

        # ---- revenue columns ---- #
        # actual_revenue_row_total = energy P&L at actual spot prices + demand charges.
        # This is total realised P&L and is directly comparable across both directives.
        #
        # For directive 1 (perfect foresight) this equals objective_row_total exactly,
        # because the LP optimised at actual prices and curtailment_penalty / signal
        # penalties are both zero.
        #
        # For directive 2 the LP optimised at a price proxy, so objective_row_total
        # (LP objective at predicted prices) differs from actual_revenue_row_total
        # (real-world cash outcome at actual spot prices). The gap between the two
        # measures how much value was lost/gained due to imperfect price prediction.
        #
        # Demand charges (network tariffs) are included via
        # objective_component_monthly_import_export_penalties, which holds the solved
        # LP demand cost allocated to the first interval of each month.

        def _add_revenue_columns():
            actual_rev = _compute_actual_revenue()

            # Add demand charges so actual_revenue_row_total is total P&L, not just
            # spot-market P&L. The demand charge column is negative (it's a cost).
            demand_col = "objective_component_monthly_import_export_penalties"
            if demand_col in extra_df.columns:
                actual_rev = actual_rev + extra_df[demand_col].fillna(0).to_numpy()

            if variables.optimisation_directive == 2 and variables.predicted_price_type == 2:
                # BESS instructions only: LP price was 0 so the LP objective captures
                # signal penalties only — rename to avoid confusing it with cash revenue
                extra_df.rename(
                    columns={"objective_row_total": "signal_penalty_row_total"},
                    inplace=True,
                )

            extra_df["actual_revenue_row_total"] = actual_rev

        _add_revenue_columns()

        return pd.concat([base_df, extra_df], axis=1).copy()

    # ------------------------------------------------------------------ #
    # Lean export — multiple (parallel) optimisations                     #
    # ------------------------------------------------------------------ #

    def _export_multiple():
        """Minimal DataFrame for parallel/batch runs.
        Provides only what the window scheduler and downstream aggregation need:
          decision_variable_stored_charge  — SoC hand-off between windows
          objective_row_total              — LP objective per interval
          actual_revenue_row_total         — real-world revenue per interval (spot P&L + demand charges)
        """
        n = len(self.dataset_window_subset)
        stored_charge = [value(v) for v in self.stored_charge]

        objective_row_total = [0.0] * n
        demand_charges = [0.0] * n
        for component_name, interval_exprs in self.objective_component_map.items():
            for i, expr in interval_exprs.items():
                if i is not None and 0 <= i < n:
                    v = value(expr)
                    objective_row_total[i] += v
                    if component_name == "monthly_import_export_penalties":
                        demand_charges[i] += v

        actual_rev = _compute_actual_revenue() + np.array(demand_charges)

        return pd.DataFrame({
            "decision_variable_stored_charge": stored_charge,
            "objective_row_total": objective_row_total,
            "actual_revenue_row_total": actual_rev,
        })

    # ------------------------------------------------------------------ #
    # Dispatch                                                             #
    # ------------------------------------------------------------------ #

    if variables.run_multiple_optimisations:
        result = _export_multiple()
    else:
        result = _export_single()

    self.optimization_window_results = result
  
    return result


def export_combination_summary(df, price_col, bess_duration_h):
    """Aggregate a completed combination's per-interval DataFrame into a single summary row."""
    return pd.DataFrame([{
        "price_col": price_col,
        "bess_duration_h": bess_duration_h,
        "sum_objective_row_total": round(df["objective_row_total"].sum(), 4),
        "sum_actual_revenue_row_total": round(df["actual_revenue_row_total"].sum(), 4),
    }])


### Interval level logic

In [18]:
def _interval_level_logic(self, i):

    # Dataset, quick reference
    gen = self.dataset_window_subset['Generation'].iloc[i]

    # Default signal and penalty values; overridden below when using BESS instructions
    charge_signal = False
    discharge_signal = False
    charge_signal_penalty = 0
    discharge_signal_penalty = 0

    # Perfect foresight
    if variables.optimisation_directive == 1:
        price = self.dataset_window_subset["Price"].iloc[i]
    
    # Price prediction
    else:
        # Predicted price only
        if variables.predicted_price_type == 1:
            price = self.dataset_window_subset["Price"].iloc[i]

        # BESS instruction only
        elif variables.predicted_price_type == 2:
            price = 0.0
            charge_signal = (self.dataset_window_subset["BESS_instruction"][i]) == 1
            discharge_signal = (self.dataset_window_subset["BESS_instruction"][i]) == 2

        # Both
        elif variables.predicted_price_type == 3:
            price = self.dataset_window_subset["Price"].iloc[i]
            charge_signal = (self.dataset_window_subset["BESS_instruction"][i]) == 1
            discharge_signal = (self.dataset_window_subset["BESS_instruction"][i]) == 2

    lgc_price = self.dataset_window_subset["LGC"].iloc[i]
    import_charge = self.dataset_window_subset["Import_charge"][i]
    tou_label = self.dataset_window_subset["ToU_label"].iloc[i]
    gen_exportable = self.dataset_window_subset['Generation_exportable'][i]
    
    # Decision variables, quick reference
    pv2grid = self.pv2grid[i]
    pv2battery = self.pv2battery[i]
    curtailment = self.curtailment[i]
    grid2battery = self.grid2battery[i]
    dischargeE = self.dischargeE[i]
    stored_charge = self.stored_charge[i]

    year_month = self.window_year_month[i] 
    imports_max_peak = self.imports_max_peak[year_month]
    imports_max_shoulder = self.imports_max_shoulder[year_month]
    imports_max_offpeak = self.imports_max_offpeak[year_month]
    exports_max_sunsoaker = self.exports_max_sunsoaker[year_month]
    
    # Other calculated varibles, quick reference
    chargeE = self.pv2battery[i] + self.grid2battery[i]
    pv2battery_eff = pv2battery * self.dataset_window_subset["hte_from_pv"].iloc[i]
    grid2battery_eff = grid2battery * self.dataset_window_subset["hte_from_grid"].iloc[i]
    dischargeE_eff = dischargeE * self.dataset_window_subset["hte2grid"].iloc[i]

    def _previous_interval():
        previous_window_ending_mwh = self.window_start_soc * self.bess_mwh
        previous_interval_mwh = previous_window_ending_mwh if i == 0 else self.stored_charge[i - 1]
        return previous_interval_mwh
    
    previous_interval_mwh = _previous_interval()

    bess_max_eff = self.bess_mwh * self.dataset_window_subset["bess_deg"].iloc[i]
    bess_capacity_remaining = bess_max_eff - self.stored_charge[i]

    gen_exportable_now = self.dataset_window_subset['Generation_exportable'][i] > 0
    sun_soaker = self.dataset_window_subset["ToU_label"].iloc[i] != "Sun Soaker"

    
    # Call constraints
    self.Lp = _interval_constraint_A(self.Lp, gen, pv2grid, pv2battery, curtailment, i)
    self.constraint_map[f"Ballance_PV_output_{i}"] = i

    self.Lp = _interval_constraint_B(self.Lp, pv2battery_eff, grid2battery_eff, dischargeE, previous_interval_mwh, stored_charge, i)
    self.constraint_map[f"Ballance_BESS_charge_{i}"] = i

    self.Lp = _interval_constraint_C(self.Lp, pv2battery_eff, grid2battery_eff, dischargeE_eff, self.bess_energy_limit_per_interval_mw, i)
    self.constraint_map[f"Restrict_BESS_by_BESS_capacity_{i}"] = i
    
    self.Lp = _interval_constraint_D(self.Lp, stored_charge, bess_max_eff, chargeE, dischargeE, gen_exportable_now, pv2battery, gen_exportable, i)
    self.constraint_map[f"Restrict_stored_charge_by_physcial_limit_{i}"] = i
    self.constraint_map[f"Restrict_chargeE_by_physcial_limit_{i}"] = i
    self.constraint_map[f"Restrict_dischargeE_by_physcial_limit_{i}"] = i

    self.Lp = _interval_constraint_E(self.Lp, pv2grid, dischargeE_eff, variables.project_poi_export_limit_mwh_per_interval, grid2battery, variables.project_poi_import_limit_mwh_per_interval, i)
    self.constraint_map[f"Restrict_grid_by_grid_capacity_{i}"] = i

    if variables.optimisation_directive == 2 and variables.predicted_price_type != 1:
        self.Lp = _interval_constraint_F(self.Lp, charge_signal, discharge_signal, chargeE, dischargeE, i)
        self.constraint_map[f"Restrict_BESS_by_dischargeE_mutuality_{i}"] = i
        self.constraint_map[f"Restrict_BESS_by_chargeE_mutuality_{i}"] = i

    # self.Lp = _interval_constraint_G(self.Lp, sun_soaker, grid2battery, i)
    # self.constraint_map[f"Restrict_importing_by_sunsoaker_{i}"] = i

    self.Lp = _interval_constraint_H(self.Lp, tou_label, grid2battery, imports_max_peak, imports_max_shoulder, imports_max_offpeak, exports_max_sunsoaker, dischargeE_eff, pv2grid, i)
    self.constraint_map[f"Restrict_onpeak_grid_imports_by_max_{i}"] = i
    self.constraint_map[f"Restrict_shoulder_grid_imports_by_max_{i}"] = i
    self.constraint_map[f"Restrict_peak_grid_imports_by_max_{i}"] = i
    self.constraint_map[f"Restrict_sunsoaker_grid_exports_by_max_{i}"] = i

    # Call penalties
    if variables.optimisation_directive == 2 and variables.predicted_price_type != 1:
        charge_signal_penalty = _interval_penalty_A(bess_capacity_remaining, charge_signal, variables.trade_signal_penalty)
        discharge_signal_penalty = _interval_penalty_B(discharge_signal, variables.trade_signal_penalty, stored_charge)

    curtailed_energy_penalty = _interval_penalty_C(curtailment, variables.curtailment_penalty)

    # Call objective expression
    interval_expression, interval_objective_components = _interval_expression(price, variables.grid_import_penalty, lgc_price, grid2battery, pv2grid, dischargeE_eff, import_charge, charge_signal, discharge_signal, charge_signal_penalty, discharge_signal_penalty, curtailed_energy_penalty)
    for component_name, component_expr in interval_objective_components.items():
        self.objective_component_map.setdefault(component_name, {})[i] = component_expr
    self.interval_objective_expressions.append(interval_expression)

    return self


### Window level logic

In [19]:
def _window_level_logic(self):
    self.Lp = _window_constraint_A(self.Lp, self.window_cycle_budget_mwh, lpSum([self.dischargeE[i] for i in self.dataset_index_range]))

    return self

### Within-window controller

In [20]:
class Optimizer():

    def __init__(self, dataset_window_subset:pd.DataFrame, window_start_soc:float, dataset_index_range:range, bess_mwh, bess_energy_limit_per_interval_mw):
        self.dataset_window_subset = dataset_window_subset
        self.window_start_soc = window_start_soc
        self.dataset_index_range = dataset_index_range
        self.interval_objective_expressions = []
        self.monthly_import_export_penalties = []
        self.objective_component_map = {}
        self.constraint_map = {}

        self.bess_mwh = bess_mwh
        self.bess_energy_limit_per_interval_mw = bess_energy_limit_per_interval_mw

        self.window_months, self.window_year_month = self._calculate_years_months()
        self.window_cycle_budget_mwh = self._calculate_cycle_budget()
        self._main_loop()
    

    def _calculate_years_months(self):
        window_dates = pd.to_datetime(self.dataset_window_subset["Date"])
        window_year_month = list(zip(window_dates.dt.year.to_numpy(), window_dates.dt.month.to_numpy()))
        window_months = sorted(set(window_year_month))
        return window_months, window_year_month
        
    def _calculate_cycle_budget(self):
        periods_per_day = 24 * (60 / variables.operation_granularity_in_minutes)
        days_in_window = len(self.dataset_window_subset) / periods_per_day
        avg_window_deg = np.mean(self.dataset_window_subset["bess_deg"])
        window_cycle_budget_mwh = variables.cycles_per_day * self.bess_mwh * avg_window_deg * days_in_window
        return window_cycle_budget_mwh
    
    def _main_loop(self):
        # Define Linear Programming optimization type
        self.Lp = LpProblem("Optimization", LpMaximize)
        
        # Define decision variables
        self.stored_charge = [LpVariable(f"stored_charge_{i}", lowBound=0) for i in self.dataset_index_range]
        self.pv2grid = [LpVariable(f"pv2grid_{i}", lowBound=0) for i in self.dataset_index_range]
        self.pv2battery = [LpVariable(f"pv2battery_{i}", lowBound=0) for i in self.dataset_index_range]
        self.curtailment = [LpVariable(f"curtailment_{i}", lowBound=0) for i in self.dataset_index_range]
        self.dischargeE = [LpVariable(f"dischargeE_{i}", lowBound=0) for i in self.dataset_index_range]
        self.grid2battery = [LpVariable(f"grid2battery_{i}", lowBound=0, upBound=variables.project_poi_import_limit_mwh_per_interval,) for i in self.dataset_index_range]
        self.imports_max_peak = {ym: LpVariable(f"imports_max_peak_{ym[0]}_{ym[1]:02d}", lowBound=0) for ym in self.window_months}
        self.imports_max_shoulder = {ym: LpVariable(f"imports_max_shoulder_{ym[0]}_{ym[1]:02d}", lowBound=0) for ym in self.window_months}
        self.imports_max_offpeak = {ym: LpVariable(f"imports_max_offpeak_{ym[0]}_{ym[1]:02d}", lowBound=0) for ym in self.window_months}
        self.exports_max_sunsoaker = {ym: LpVariable(f"exports_max_sunsoaker_{ym[0]}_{ym[1]:02d}", lowBound=0) for ym in self.window_months}
     
        # Create per month objective penalties here
        for ym in self.window_months:

            monthly_import_export_penalty = _monthly_penalty_A(variables.demand_charge_peak,variables.demand_charge_shoulder,variables.demand_charge_off_peak,variables.export_charge_sun_soaker,self.imports_max_peak[ym],self.imports_max_shoulder[ym],self.imports_max_offpeak[ym],self.exports_max_sunsoaker[ym])
            self.monthly_import_export_penalties.append(monthly_import_export_penalty)
            first_interval_of_month = self.window_year_month.index(ym)
            self.objective_component_map.setdefault("monthly_import_export_penalties", {})[first_interval_of_month] = -monthly_import_export_penalty

        # Create per INTERVAL constraints and objectives
        for i in self.dataset_index_range:
            _interval_level_logic(self, i)

        # Create per WINDOW constraints and objectives
        _window_level_logic(self)

        # Create objective
        self.Lp = _objective(self)

        # Solve
        self.model = self.Lp.solve(PULP_CBC_CMD(msg=0))


## Window scheduler logic:

In [21]:
def create_optimisation(inputs, item=None):

    # get the data the normal way, through the passed dataset
    def get_inputs_single_series():
        dataset = inputs["dataset"]
        bess_mwh = variables.bess_mwh
        bess_energy_limit_per_interval_mw = variables.bess_energy_limit_per_interval_mw

        return dataset, bess_mwh, bess_energy_limit_per_interval_mw
    
    # modify the dataset with new item conbination data based on this passed item
    def get_inputs_multi_series():
        dataset = inputs["dataset"]
        multi_series_price = inputs["multi_series_price"]
        multi_series_bess = inputs["multi_series_bess"]
        price_col,bess_col = item

        dataset["Price"] = multi_series_price[price_col]
        
        bess_mwh = multi_series_bess.iloc[bess_col]["BESS_MWh"]
        bess_energy_limit_per_interval_mw = multi_series_bess.iloc[bess_col]["BESS_MWh_per_interval"]

        bess_row = multi_series_bess.iloc[bess_col]

        return dataset, bess_mwh, bess_energy_limit_per_interval_mw, price_col, bess_row

    if variables.run_multiple_optimisations == False:
        dataset, bess_mwh, bess_energy_limit_per_interval_mw = get_inputs_single_series()
    else:
        dataset, bess_mwh, bess_energy_limit_per_interval_mw, price_col, bess_row = get_inputs_multi_series()


    length_of_single_lookahead_window = int(int(variables.optimization_avoid_edge_effect_total_hours) * (60 / int(variables.operation_granularity_in_minutes)))
    length_of_window_minus_lookahead = int(int(variables.number_of_intervals_per_window) - length_of_single_lookahead_window)

    retained_outputs = []
    window_start_soc = variables.start_soc
    window_interval_stride = length_of_window_minus_lookahead

    if variables.display_window_scheduler_visual and not variables.run_multiple_optimisations:
        # Visualisation purposes only
        window_starts = list(range(0, len(dataset), window_interval_stride))
        intervals_per_bar = length_of_single_lookahead_window
        bars_per_window = int(variables.number_of_intervals_per_window / intervals_per_bar)
        retained_bars_per_window = int(length_of_window_minus_lookahead / intervals_per_bar)
        fig, completed_x, completed_y, completed_colors = create_window_progress_visual(
            window_starts=window_starts,
            total_dataset_intervals=len(dataset),
            intervals_per_bar=intervals_per_bar,
            bars_per_window=bars_per_window,
            retained_bars_per_window=retained_bars_per_window,
            dataset=dataset,
        )
        
    for i, start_interval in enumerate(range(0, len(dataset), window_interval_stride)):

        stop_interval = min(start_interval + variables.number_of_intervals_per_window, len(dataset))
        dataset_window_subset = dataset.iloc[start_interval:stop_interval].copy().reset_index(drop=True)
        dataset_index_range = range(len(dataset_window_subset))
        
        Optimization = Optimizer(dataset_window_subset=dataset_window_subset, window_start_soc=window_start_soc, dataset_index_range=dataset_index_range, bess_mwh=bess_mwh, bess_energy_limit_per_interval_mw=bess_energy_limit_per_interval_mw)

        optimization_output = export_results(Optimization)
        
        optimization_output["Window"] = i

        if stop_interval == len(dataset):
            retained_optimization_output = optimization_output
        else:
            retained_optimization_output = optimization_output.iloc[:length_of_window_minus_lookahead].copy()
            last_retained_stored_charge_mwh = optimization_output["decision_variable_stored_charge"].iloc[length_of_window_minus_lookahead - 1]
            window_start_soc = last_retained_stored_charge_mwh / bess_mwh

        retained_outputs.append(retained_optimization_output)

        if variables.display_window_scheduler_visual and not variables.run_multiple_optimisations:
            # Visualisation purposes only
            actual_bars_this_window = min(bars_per_window, int((stop_interval - start_interval) / intervals_per_bar))
            if actual_bars_this_window > 0:
                update_window_progress_visual(fig=fig, current_window=i, current_horizon_bar=actual_bars_this_window - 1, window_starts=window_starts, bars_per_window=bars_per_window, completed_x=completed_x, completed_y=completed_y, completed_colors=completed_colors, force=(stop_interval == len(dataset)))

    retained_outputs = pd.concat(retained_outputs, ignore_index=True, sort=False)
    retained_outputs = retained_outputs.round(4)
    
    if variables.run_multiple_optimisations:
        retained_outputs = export_combination_summary(retained_outputs, price_col, bess_row["BESS_Duration_h"])

    return retained_outputs


## Prepare optimization inputs (Code entry point)

In [22]:
def prepare_inputs():

    row_limit = 8760
    price_series_limit = 1000
    bess_series_limit = None

    # dataset - core dataset contain data to optimizer over
    dataset = pd.read_csv("../1_Dataset/2_Processed_data/Dataset/dataset.csv")
    dataset["Generation_exportable"] = np.clip(
        dataset["Generation"] - variables.project_poi_export_limit_mwh_per_interval,
        a_min=0,a_max=None
    )
    dataset = dataset[:row_limit]

    if variables.optimisation_directive == 1:
        # multi_series_price_actual 
        multi_series_price_actual = pd.read_csv("../1_Dataset/2_Processed_data/resampled_aurora_price.csv")
        operation_start_date = pd.to_datetime(variables.operation_start_date, format="%d/%m/%Y %H:%M")
        operation_end_date = pd.to_datetime(variables.operation_end_date, format="%d/%m/%Y %H:%M")
        multi_series_price_actual["Date"] = pd.to_datetime(multi_series_price_actual["Date"])
        multi_series_price_actual = multi_series_price_actual.loc[(multi_series_price_actual["Date"] >= operation_start_date)& (multi_series_price_actual["Date"] <= operation_end_date)].copy()
        multi_series_price_actual = multi_series_price_actual.drop(columns=["Date"]).reset_index(drop=True)
        
        multi_series_price = multi_series_price_actual.iloc[:row_limit,:price_series_limit]
    else:
        # multi_series_price_predicted 
        multi_series_price_predicted = pd.read_csv("../1_Dataset/2_Processed_data/predicted_prices.csv")
        multi_series_price_predicted = multi_series_price_predicted.drop(columns=["Date"])

        multi_series_price = multi_series_price_predicted.iloc[:row_limit,:price_series_limit]
    
    # multi_series_bess
    multi_series_bess = pd.read_csv("../1_Dataset/2_Processed_data/bess_designs.csv")
    multi_series_bess = multi_series_bess.iloc[:bess_series_limit]

    return {
        "dataset": dataset,
        "multi_series_price": multi_series_price,
        "multi_series_bess": multi_series_bess,
    }

inputs = prepare_inputs()

dataset = inputs["dataset"]
multi_series_price = inputs["multi_series_price"]
multi_series_bess = inputs["multi_series_bess"]

In [23]:
dataset[:5]

,Date,Generation,LGC,Import_charge,ToU_label,bess_deg,hte_from_pv,hte_from_grid,hte2grid,pv_bess_grid,grid_bess_grid,Generation_exportable
0,2026-01-01 00:00:00,0.0,6.48,0.0,Off-peak,1.000000,0.926727,0.880467,0.942151,0.851046,0.829534,0.0
1,2026-01-01 01:00:00,0.0,6.48,0.0,Off-peak,0.999995,0.926726,0.880467,0.942151,0.851046,0.829533,0.0
2,2026-01-01 02:00:00,0.0,6.48,0.0,Off-peak,0.999991,0.926726,0.880467,0.942151,0.851046,0.829533,0.0
3,2026-01-01 03:00:00,0.0,6.48,0.0,Off-peak,0.999986,0.926726,0.880466,0.942151,0.851045,0.829533,0.0
4,2026-01-01 04:00:00,0.0,6.48,0.0,Off-peak,0.999982,0.926725,0.880466,0.942151,0.851045,0.829532,0.0


In [24]:
multi_series_price[:5]

,Sample 1,Sample 2,Sample 3,Sample 4,Sample 5,Sample 6,Sample 7,Sample 8,Sample 9,Sample 10,Sample 11,Sample 12,Sample 13,Sample 14,Sample 15,Sample 16,Sample 17,Sample 18,Sample 19,Sample 20,Sample 21,Sample 22,Sample 23,Sample 24,Sample 25,Sample 26,Sample 27,Sample 28,Sample 29,Sample 30,Sample 31,Sample 32,Sample 33,Sample 34,Sample 35,Sample 36,Sample 37,Sample 38,Sample 39,Sample 40,Sample 41,Sample 42,Sample 43,Sample 44,Sample 45,Sample 46,Sample 47,Sample 48,Sample 49,Sample 50,Sample 51,Sample 52,Sample 53,Sample 54,Sample 55,Sample 56,Sample 57,Sample 58,Sample 59,Sample 60,Sample 61,Sample 62,Sample 63,Sample 64,Sample 65,Sample 66,Sample 67,Sample 68,Sample 69,Sample 70,Sample 71,Sample 72,Sample 73,Sample 74,Sample 75,Sample 76,Sample 77,Sample 78,Sample 79,Sample 80,Sample 81,Sample 82,Sample 83,Sample 84,Sample 85,Sample 86,Sample 87,Sample 88,Sample 89,Sample 90,Sample 91,Sample 92,Sample 93,Sample 94,Sample 95,Sample 96,Sample 97,Sample 98,Sample 99,Sample 100,Sample 101,Sample 102,Sample 103,Sample 104,Sample 105,Sample 106,Sample 107,Sample 108,Sample 109,Sample 110,Sample 111,Sample 112,Sample 113,Sample 114,Sample 115,Sample 116,Sample 117,Sample 118,Sample 119,Sample 120,Sample 121,Sample 122,Sample 123,Sample 124,Sample 125,Sample 126,Sample 127,Sample 128,Sample 129,Sample 130,Sample 131,Sample 132,Sample 133,Sample 134,Sample 135,Sample 136,Sample 137,Sample 138,Sample 139,Sample 140,Sample 141,Sample 142,Sample 143,Sample 144,Sample 145,Sample 146,Sample 147,Sample 148,Sample 149,Sample 150,Sample 151,Sample 152,Sample 153,Sample 154,Sample 155,Sample 156,Sample 157,Sample 158,Sample 159,Sample 160,Sample 161,Sample 162,Sample 163,Sample 164,Sample 165,Sample 166,Sample 167,Sample 168,Sample 169,Sample 170,Sample 171,Sample 172,Sample 173,Sample 174,Sample 175,Sample 176,Sample 177,Sample 178,Sample 179,Sample 180,Sample 181,Sample 182,Sample 183,Sample 184,Sample 185,Sample 186,Sample 187,Sample 188,Sample 189,Sample 190,Sample 191,Sample 192,Sample 193,Sample 194,Sample 195,Sample 196,Sample 197,Sample 198,Sample 199,Sample 200,Sample 201,Sample 202,Sample 203,Sample 204,Sample 205,Sample 206,Sample 207,Sample 208,Sample 209,Sample 210,Sample 211,Sample 212,Sample 213,Sample 214,Sample 215,Sample 216,Sample 217,Sample 218,Sample 219,Sample 220,Sample 221,Sample 222,Sample 223,Sample 224,Sample 225,Sample 226,Sample 227,Sample 228,Sample 229,Sample 230,Sample 231,Sample 232,Sample 233,Sample 234,Sample 235,Sample 236,Sample 237,Sample 238,Sample 239,Sample 240,Sample 241,Sample 242,Sample 243,Sample 244,Sample 245,Sample 246,Sample 247,Sample 248,Sample 249,Sample 250,Sample 251,Sample 252,Sample 253,Sample 254,Sample 255,Sample 256,Sample 257,Sample 258,Sample 259,Sample 260,Sample 261,Sample 262,Sample 263,Sample 264,Sample 265,Sample 266,Sample 267,Sample 268,Sample 269,Sample 270,Sample 271,Sample 272,Sample 273,Sample 274,Sample 275,Sample 276,Sample 277,Sample 278,Sample 279,Sample 280,Sample 281,Sample 282,Sample 283,Sample 284,Sample 285,Sample 286,Sample 287,Sample 288,Sample 289,Sample 290,Sample 291,Sample 292,Sample 293,Sample 294,Sample 295,Sample 296,Sample 297,Sample 298,Sample 299,Sample 300,Sample 301,Sample 302,Sample 303,Sample 304,Sample 305,Sample 306,Sample 307,Sample 308,Sample 309,Sample 310,Sample 311,Sample 312,Sample 313,Sample 314,Sample 315,Sample 316,Sample 317,Sample 318,Sample 319,Sample 320,Sample 321,Sample 322,Sample 323,Sample 324,Sample 325,Sample 326,Sample 327,Sample 328,Sample 329,Sample 330,Sample 331,Sample 332,Sample 333,Sample 334,Sample 335,Sample 336,Sample 337,Sample 338,Sample 339,Sample 340,Sample 341,Sample 342,Sample 343,Sample 344,Sample 345,Sample 346,Sample 347,Sample 348,Sample 349,Sample 350,Sample 351,Sample 352,Sample 353,Sample 354,Sample 355,Sample 356,Sample 357,Sample 358,Sample 359,Sample 360,Sample 361,Sample 362,Sample 363,Sample 364,Sample 365,Sample 366,Sample 367,Sample 368,Sample 369,Sample 370,Sample 371,Sample 372,Sample 373,Samp

In [25]:
multi_series_bess[:5]

,BESS_Duration_h,BESS_Power_MW,BESS_MWh,BESS_MWh_per_interval
0,2.0,4.96,9.3820,4.96
1,2.5,4.96,11.7275,4.96
2,3.0,4.96,14.0730,4.96
3,3.5,4.96,16.4185,4.96
4,4.0,4.96,18.7639,4.96


## Singular / parrallel coordinator

In [26]:
# single series operating mode
if variables.run_multiple_optimisations == False:
    results = create_optimisation(inputs)
    print(results["actual_revenue_row_total"].sum())

# multi series operating mode
if variables.run_multiple_optimisations == True:

    def create_dataset_combinations():
        
        # Assign combinations of items
        items = [
            (price_col, bess_col)
            for price_col in multi_series_price.columns
            for bess_col in range(len(multi_series_bess))
        ]

        return items
    
    if variables.execution_mode == "sequential":   
        function = partial(create_optimisation, inputs)

        results = []

        for item in tqdm(create_dataset_combinations()):
            results.append(function(item))

        results = pd.concat(results, ignore_index=True)

    if variables.execution_mode == "parallel":

        data=create_dataset_combinations()
        function = partial(create_optimisation, inputs)

        results = pd.concat(run_parallel(
            function=function,
            data=data
        ), ignore_index=True)


results.to_csv("Optimization_data/optimization_results.csv", index=False)


os=linux cpu_count=48 max_assigned_workers=40


Processing..: 100%|██████████| 13000/13000 [47:01<00:00,  4.61item/s] 
